# 09 · Indicador redefinido — alertas-cd-3.0

**Ejecuta este notebook primero.** Usa el Gold existente: no descarga datos ni reconstruye Silver. El resultado v2 se conserva. Este indicador es descriptivo y retrospectivo; no está entrenado con sanciones y su puntaje no es una probabilidad de corrupción.

## Qué se calcula y por qué

| Señal | Cálculo | Umbral inicial y precaución |
|---|---|---|
| C1 | Contratos entidad–proveedor / contratos entidad, últimos 365 días incluidos el día observado | >40 %, mínimo 10 contratos de entidad. Conteos, no montos. |
| C2 | HHI: suma del cuadrado de las participaciones por proveedor en el año observado | >0,25; mínimo 10 contratos. Es una característica de entidad-año, no evidencia individual. Año en curso parcial señalado en `anio_completo`. |
| C3 | Contratos entidad–proveedor / contratos proveedor, últimos 365 días | >80 %, mínimo 5 contratos del proveedor. Dependencia dentro de contratación directa observada. |
| V1 | Z robusto del logaritmo del valor frente al mismo tipo, causal y UNSPSC de 4 dígitos | >3,5; mínimo 30 referencias. Se usan los dos años calendario completos anteriores; MAD cero usa percentil 95. |
| V2 | Cantidad entidad–proveedor–tipo–UNSPSC de 4 dígitos en 30 días | >=3. Repetición temporal, no fraccionamiento probado. |
| P1 | Fecha de primera aparición observada del proveedor | Solo evaluable con al menos 365 días de cobertura previa. No equivale a creación reciente. |
| P2 | Número de segmentos UNSPSC (2 dígitos) del proveedor en 365 días | >=3 segmentos, mínimo 5 contratos. Diversidad, no incompatibilidad probada. |

**Umbrales configurables y provisionales:** son hipótesis operativas que deben contrastarse con evidencia externa. No se presentan como umbrales validados por los artículos. C4, T1/T2/T3 y P3 quedan fuera del puntaje. La red complementaria describe grado, cantidad de contratos y actividad semanal, sin sumar riesgo.

## Cambios metodológicos frente a v2

- V1 excluye nulos, no finitos, no positivos, valores superiores a 10 billones y superiores a 1.000 veces la mediana del tipo/año. Esa mediana usa solo valores positivos, finitos y bajo el tope. Se guarda un motivo prioritario por fila, sin adivinar ni corregir el importe.
- Los inválidos no participan en referencias V1. C1/C2/C3 usan conteos y no se eliminan contratos por montos inválidos.
- La referencia V1 cambia de 24 meses móviles a **dos años completos anteriores**, para compartir cálculos y reducir costo. No es una réplica exacta de D2. No se usa el año actual como referencia V1.
- C2 y la validación por tipo/año usan información anual observada: **no usar estos resultados como predicción disponible al firmar el contrato**.
- Se usa DuckDB con memoria acotada y un hilo; Dask continúa en las etapas ETL originales. No se crean clusters nuevos para este cálculo.


In [1]:
from pathlib import Path
import sys,json
from IPython.display import display,Markdown
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'secop_pipeline').is_dir()),None)
if ROOT is None:raise RuntimeError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from secop_pipeline.indicador_v3 import configuration
GOLD=ROOT/'data/gold/contratos_control/contratos.parquet'
CONFIG=ROOT/'indicador_v3_config.json'
cfg=configuration(CONFIG,GOLD)
display(json.loads(CONFIG.read_text(encoding='utf-8')))
print('Corte declarado:',cfg.cutoff,'Inicio de cobertura:',cfg.coverage_start)


{'version': 'alertas-cd-3.0',
 'cutoff': None,
 'coverage_start': None,
 'memory_mb': 2048,
 'threads': 1,
 'history_days': 365,
 'min_entity_contracts': 10,
 'min_supplier_contracts': 5,
 'c1_threshold': 0.4,
 'c2_threshold': 0.25,
 'c3_threshold': 0.8,
 'absolute_value_cap': 10000000000000,
 'median_value_factor': 1000,
 'v1_reference_years': 2,
 'v1_min_reference': 30,
 'v1_z_threshold': 3.5,
 'v2_days': 30,
 'v2_min_contracts': 3,
 'p2_min_segments': 3,
 'review_top_k_per_quarter': 2000,
 'minimum_priority_score': 0.0}

Corte declarado: 2026-09-21 Inicio de cobertura: 2017-01-01


## Puntaje, información incompleta y alerta

Con las siete señales disponibles:

`C = (C1+C2+C3)/3`, `V = (V1+V2)/2`, `P = (P1+P2)/2`.

**Puntaje = (C+V+P)/3**. Tres familias con igual peso; no se crea una familia temporal vacía. C1/C2/C3 pesan 1/9 cada una; V1/V2/P1/P2 pesan 1/6 cada una.

La versión inicial exige las **7 señales evaluables** para un puntaje comparable. Si falta una: puntaje y alerta binaria NULL, clasificación NO_EVALUABLE. Se conservan intervalos `puntaje_min` y `puntaje_max` rellenando lo desconocido con 0 y 1 solo para acotar incertidumbre; esos límites no convierten la señal desconocida en ausencia/presencia observada.

Esta política puede dejar muchos proveedores con poca historia como NO_EVALUABLES, incluidos los nuevos detectados por P1 que no cumplen mínimos de C3/P2. Se debe revisar la cobertura antes de interpretar el ranking; no ocultamos ese efecto ni cambiamos los mínimos automáticamente.

Se priorizan hasta **2.000 contratos por trimestre**, solo entre puntajes evaluables y positivos. Empates: orden por ID, decisión operativa sin significado de riesgo. Los restantes evaluables quedan NO_ALERTA; no significa libres de riesgo. Cambia el cupo en la configuración según capacidad real. No se garantiza llenar el cupo si no hay suficientes candidatos.


In [2]:
from flows.indicador_v3_flow import indicador_v3_flow
resultado=indicador_v3_flow(config_path=str(CONFIG),source=str(GOLD))
print('Estado:',resultado['estado'])
print('Directorio:',resultado['directorio'])
display(resultado['resumen'])


06:25:07.812 | INFO    | Flow run 'rich-myna' - Beginning flow run 'rich-myna' for flow 'SECOP-INDICADOR-V3-DESDE-GOLD'

06:25:07.829 | INFO    | Flow run 'rich-myna' - View at https://app.prefect.cloud/account/645e4433-1321-4cfd-9a35-a54cbed1b6d6/workspace/e1e6136c-4435-4483-bb86-c454bd7494a4/runs/flow-run/06adb441-294e-7acf-8000-d0a69fd687b9

06:25:10.919 | INFO    | Flow run 'rich-myna' - 1/5 preparar_gold: iniciando

06:25:26.320 | INFO    | Task run 'indicador-v3-etapa-47b' - Finished in state Completed()

06:25:26.326 | INFO    | Flow run 'rich-myna' - 2/5 relaciones: iniciando

...

06:27:31.119 | INFO    | Task run 'indicador-v3-etapa-1ba' - Finished in state Completed()

06:27:46.289 | INFO    | Flow run 'rich-myna' - {
  "contratos": 4353720,
  "clasificaciones": {
    "NO_EVALUABLE": 4125049,
    "NO_ALERTA": 186084,
    "ALERTA": 42587
  },
  "dimensiones": {
    "C1": {
      "total": 4353720,
      "evaluados": 4330105,
      "alertas": 206,
      "sin_senal": 4329899,
      "no_evaluables": 23615
    },
    "C2": {
      "total": 4353720,
      "evaluados": 4343765,
      "alertas": 369,
      "sin_senal": 4343396,
      "no_evaluables": 9955
    },
    "C3": {
      "total": 4353720,
      "evaluados": 262261,
      "alertas": 69082,
      "sin_senal": 193179,
      "no_evaluables": 4091459
    },
    "V1": {
      "total": 4353720,
      "evaluados": 4246400,
      "alertas": 13340,
      "sin_senal": 4233060,
      "no_evaluables": 107320
    },
    "V2": {
      "total": 4353720,
      "evaluados": 4332972,
      "alertas": 13057,
      "sin_senal": 4319915,
      "no_evaluables": 20748
    },
    "P1": {
      "total": 4353720,
      "evaluados": 4349673,
      "alertas": 1024545,
      "sin_senal": 3325128,
      "no_evaluables": 4047
    },
    "P2": {
      "total": 4353720,
      "evaluados": 248202,
      "alertas": 57264,
      "sin_senal": 190938,
      "no_evaluables": 4105518
    }
  },
  "valores_invalidos": {
    "no_positivo": 35664,
    "supera_factor_mediana": 411,
    "supera_tope_absoluto": 23
  }
}

06:27:46.891 | INFO    | Flow run 'rich-myna' - Finished in state Completed()

Estado: COMPLETADA
Directorio: C:\Users\luna3\OneDrive\Desktop\secop_pipeline_equipo\salida\alertas\indicador_v3\f8045bc1b518250ab57fc739


{'contratos': 4353720,
 'clasificaciones': {'NO_EVALUABLE': 4125049,
  'NO_ALERTA': 186084,
  'ALERTA': 42587},
 'dimensiones': {'C1': {'total': 4353720,
   'evaluados': 4330105,
   'alertas': 206,
   'sin_senal': 4329899,
   'no_evaluables': 23615},
  'C2': {'total': 4353720,
   'evaluados': 4343765,
   'alertas': 369,
   'sin_senal': 4343396,
   'no_evaluables': 9955},
  'C3': {'total': 4353720,
   'evaluados': 262261,
   'alertas': 69082,
   'sin_senal': 193179,
   'no_evaluables': 4091459},
  'V1': {'total': 4353720,
   'evaluados': 4246400,
   'alertas': 13340,
   'sin_senal': 4233060,
   'no_evaluables': 107320},
  'V2': {'total': 4353720,
   'evaluados': 4332972,
   'alertas': 13057,
   'sin_senal': 4319915,
   'no_evaluables': 20748},
  'P1': {'total': 4353720,
   'evaluados': 4349673,
   'alertas': 1024545,
   'sin_senal': 3325128,
   'no_evaluables': 4047},
  'P2': {'total': 4353720,
   'evaluados': 248202,
   'alertas': 57264,
   'sin_senal': 190938,
   'no_evaluables': 4105

## Recuperación y continuación

Cada etapa finalizada queda en DuckDB y en el manifiesto. Si falla, ejecuta la misma celda: se reutilizan etapas confirmadas para los mismos bytes Gold, parámetros y código. No borres locks/checkpoints; un lock existente exige revisar la ejecución activa. Cambiar parámetros o código produce un lote nuevo bajo `salida/alertas/indicador_v3`.

El puntero `ultima_ejecucion_v3.json` se actualiza solo cuando el lote termina. Los notebooks 10, 12 y 13 rechazan la ausencia de un lote v3: nunca recurren silenciosamente a v2. Sigue con **10** (agregados), **12** (Grafana local) y **13** (AWS compacto). **11** permite observar esta ejecución desde otro kernel mientras corre.
